In [ ]:
# # OPPE Mock: Wasserstein GAN (WGAN) on MNIST

# ## 1. Initial Setup

# This cell handles all necessary imports, sets hyperparameters, and configures the environment for reproducibility.

# ### 1.1. Imports
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt

# ### 1.2. Hyperparameters & Configuration
# These parameters are chosen based on the WGAN paper and common practices for MNIST.
# Training hyperparameters
epochs = 25  # Number of training epochs
lr = 0.00005  # Learning rate for both networks (as recommended in WGAN paper)
batch_size = 64  # Batch size for training
latent_dim = 100  # Dimensionality of the latent space (z)
n_critic = 5  # Number of critic updates per generator update
clip_value = 0.01  # Weight clipping parameter

# Image and data configuration
img_size = 28  # MNIST image dimension
channels = 1  # MNIST is grayscale

# ### 1.3. Reproducibility and Device Configuration
# Set a fixed random seed for consistent results.
torch.manual_seed(42)
np.random.seed(42)

# Configure the device for training (use GPU if available).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create directory to save generated images
os.makedirs("generated_images", exist_ok=True)


In [ ]:
# ## 2. Data Loading and Preprocessing

# Here, we define the transformations, load the MNIST dataset, and create a `DataLoader`.
# The images are normalized to the `[-1, 1]` range to match the generator's `tanh` output.

# ### 2.1. Define Transformations and Load Dataset
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # Normalizes images to [-1, 1]
])

train_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

# ### 2.2. Sanity Check: Visualize a Batch of Training Data
# This step confirms that the data is loaded and normalized correctly.
real_batch = next(iter(dataloader))
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("Training Images (Sanity Check)")
plt.imshow(
    make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu().permute(1, 2, 0)
)
plt.show()


In [ ]:
# ## 3. WGAN Model Definitions

# The Generator and Critic models are defined here. These architectures are simple but effective
# for the MNIST dataset, inspired by the original DCGAN and WGAN papers.

# ### 3.1. Generator
# The Generator takes a latent vector `z` and maps it into a `28x28` grayscale image.
# It uses `ConvTranspose2d` layers to upsample the latent vector.
# The final layer uses a `tanh` activation to scale the output to `[-1, 1]`.

class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Input: latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, 256, 7, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # State: 256 x 7 x 7
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # State: 128 x 14 x 14
            nn.ConvTranspose2d(128, channels, 4, 2, 1, bias=False),
            nn.Tanh()
            # Output: channels x 28 x 28
        )

    def forward(self, input):
        return self.main(input)

# ### 3.2. Critic (Discriminator)
# The Critic takes an image and outputs a single scalar value (the "critic score").
# It does *not* use a sigmoid activation in the final layer, as it's a regressor, not a classifier.
# It also avoids `BatchNorm` in favor of other stabilization techniques if needed, but for WGAN with
# weight clipping, it's often kept simple.

class Critic(nn.Module):
    def __init__(self):
        super(Critic, self).__init__()
        self.main = nn.Sequential(
            # Input: channels x 28 x 28
            nn.Conv2d(channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # State: 64 x 14 x 14
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # State: 128 x 7 x 7
            nn.Conv2d(128, 1, 7, 1, 0, bias=False),
            # Output: 1 x 1 x 1 (a single scalar score)
        )

    def forward(self, input):
        return self.main(input).view(-1, 1)

# ### 3.3. Initialize Models and Weights
# A custom weight initialization function is used to help with training stability.

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Instantiate the models and apply the custom weight initialization
generator = Generator().to(device)
critic = Critic().to(device)

generator.apply(weights_init)
critic.apply(weights_init)

print("--- Generator Architecture ---")
print(generator)
print("\n--- Critic Architecture ---")
print(critic)


In [ ]:
# ## 4. Optimizers and Fixed Noise

# ### 4.1. Optimizers
# The WGAN paper recommends using RMSprop as the optimizer. We set it up for both the
# generator and the critic.
optimizer_G = optim.RMSprop(generator.parameters(), lr=lr)
optimizer_C = optim.RMSprop(critic.parameters(), lr=lr)

# ### 4.2. Fixed Noise for Visualization
# A fixed batch of latent vectors is created to track the generator's progress over time.
# By using the same input, we can see how the output evolves across epochs.
fixed_noise = torch.randn(64, latent_dim, 1, 1, device=device)


In [ ]:
# ## 5. WGAN Training Loop

# This is the core of the WGAN implementation. The training process alternates between
# updating the critic and the generator.

# Lists to store losses for plotting
G_losses = []
C_losses = []

print("Starting Training Loop...")
for epoch in range(epochs):
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        
        # ---------------------
        #  Train Critic
        # ---------------------
        
        # The critic is trained for `n_critic` iterations for every generator update.
        # This ensures the critic's estimates are reliable.
        optimizer_C.zero_grad()
        
        # Generate a batch of fake images
        noise = torch.randn(real_imgs.size(0), latent_dim, 1, 1, device=device)
        fake_imgs = generator(noise).detach()
        
        # Get critic scores for real and fake images
        critic_real = critic(real_imgs)
        critic_fake = critic(fake_imgs)
        
        # The WGAN loss function aims to maximize the difference between the critic's
        # output on real and fake images.
        loss_C = -(torch.mean(critic_real) - torch.mean(critic_fake))
        
        loss_C.backward()
        optimizer_C.step()
        
        # Clip the weights of the critic to enforce the Lipschitz constraint.
        # This is a key part of the original WGAN algorithm.
        for p in critic.parameters():
            p.data.clamp_(-clip_value, clip_value)
            
        # -----------------
        #  Train Generator
        # -----------------
        
        # The generator is trained less frequently than the critic.
        if i % n_critic == 0:
            optimizer_G.zero_grad()
            
            # Generate a new batch of fake images
            gen_noise = torch.randn(real_imgs.size(0), latent_dim, 1, 1, device=device)
            gen_imgs = generator(gen_noise)
            
            # Get critic scores for the generated images
            critic_gen = critic(gen_imgs)
            
            # The generator's loss is designed to maximize the critic's output for fake images,
            # which encourages the generator to produce more realistic images.
            loss_G = -torch.mean(critic_gen)
            
            loss_G.backward()
            optimizer_G.step()

            # --- Logging and Visualization ---
            if i % 100 == 0:
                print(
                    f"[Epoch {epoch+1}/{epochs}] [Batch {i}/{len(dataloader)}] "
                    f"[C loss: {loss_C.item():.4f}] [G loss: {loss_G.item():.4f}] "
                    f"[Critic Real: {critic_real.mean().item():.4f}] [Critic Fake: {critic_fake.mean().item():.4f}]"
                )
                
                # Store losses for plotting
                G_losses.append(loss_G.item())
                C_losses.append(loss_C.item())


In [ ]:
# ## 6. Image Generation and Visualization

# After each epoch, we generate a grid of images using the `fixed_noise` vector.
# This allows us to visually track the generator's improvement over time.

    with torch.no_grad():
        generator.eval() # Set generator to evaluation mode
        fake_samples = generator(fixed_noise).detach().cpu()
        generator.train() # Set back to training mode

    # Save and display the generated images
    save_image(fake_samples, f"generated_images/epoch_{epoch+1}.png", nrow=8, normalize=True)
    
    # Display the grid of generated images
    grid = make_grid(fake_samples, nrow=8, normalize=True)
    plt.figure(figsize=(8,8))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(f"Generated Images at Epoch {epoch+1}")
    plt.axis("off")
    plt.show()


In [ ]:
# ## 7. Plot Training Losses

# Finally, we plot the generator and critic losses to analyze the training dynamics.
# In a stable WGAN, the critic loss should converge towards zero, while the generator
# loss may fluctuate but should not diverge.

plt.figure(figsize=(10, 5))
plt.title("Generator and Critic Loss During Training")
plt.plot(G_losses, label="Generator Loss")
plt.plot(C_losses, label="Critic Loss")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()
